# Q1. Install Spark & PySpark

In [1]:
import os

# Set HADOOP_HOME ke folder induk dari 'bin'
os.environ['HADOOP_HOME'] = r'C:\hadoop'

# Tambahkan bin ke PATH agar Windows bisa menemukan winutils.exe
os.environ['PATH'] += os.path.pathsep + r'C:\hadoop\bin'

In [2]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("NY Taxi") \
    .config("spark.ui.port", "4040") \
    .getOrCreate()

print(f"Spark version: {spark.version}")

Spark version: 4.1.1


In [3]:
spark

# Q2 - Q4 Yellow Data Trip 2025-11

In [17]:
from pyspark.sql.types import StructType, StructField, IntegerType, LongType, DoubleType, StringType, TimestampNTZType
from pyspark.sql import functions as F

In [7]:
# get the data
!curl -o yellow_tripdata_2025-11.parquet https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0 67.8M    0  326k    0     0   763k      0  0:01:31 --:--:--  0:01:31  763k
  1 67.8M    1 1142k    0     0   802k      0  0:01:26  0:00:01  0:01:25  802k
  2 67.8M    2 1916k    0     0   792k      0  0:01:27  0:00:02  0:01:25  792k
  4 67.8M    4 2876k    0     0   839k      0  0:01:22  0:00:03  0:01:19  839k
  5 67.8M    5 3532k    0     0   799k      0  0:01:26  0:00:04  0:01:22  799k
  6 67.8M    6 4252k    0     0   785k      0  0:01:28  0:00:05  0:01:23  786k
  7 67.8M    7 5164k    0     0   805k      0  0:01:26  0:00:06  0:01:20  805k
  9 67.8M    9 6508k    0     0   877k      0  0:01:19  0:00:07  0:01:12  918k
 10 67.8M   10 7404k    0     0   879k      0  0:01:18  0:00:08  0:01:10  907k
 11 67.8M   11 8300k    0     0   881k      0  0:01

In [8]:
# read the data
df = spark.read.option("header", "true").parquet("yellow_tripdata_2025-11.parquet")
df.schema

StructType([StructField('VendorID', IntegerType(), True), StructField('tpep_pickup_datetime', TimestampNTZType(), True), StructField('tpep_dropoff_datetime', TimestampNTZType(), True), StructField('passenger_count', LongType(), True), StructField('trip_distance', DoubleType(), True), StructField('RatecodeID', LongType(), True), StructField('store_and_fwd_flag', StringType(), True), StructField('PULocationID', IntegerType(), True), StructField('DOLocationID', IntegerType(), True), StructField('payment_type', LongType(), True), StructField('fare_amount', DoubleType(), True), StructField('extra', DoubleType(), True), StructField('mta_tax', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('tolls_amount', DoubleType(), True), StructField('improvement_surcharge', DoubleType(), True), StructField('total_amount', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True), StructField('Airport_fee', DoubleType(), True), StructField('cbd_congestio

In [9]:
schema = StructType([
    StructField('VendorID', IntegerType(), True),
    StructField('tpep_pickup_datetime', TimestampNTZType(), True),
    StructField('tpep_dropoff_datetime', TimestampNTZType(), True),
    StructField('passenger_count', LongType(), True),
    StructField('trip_distance', DoubleType(), True),
    StructField('RatecodeID', LongType(), True),
    StructField('store_and_fwd_flag', StringType(), True),
    StructField('PULocationID', IntegerType(), True),
    StructField('DOLocationID', IntegerType(), True),
    StructField('payment_type', LongType(), True),
    StructField('fare_amount', DoubleType(), True),
    StructField('extra', DoubleType(), True),
    StructField('mta_tax', DoubleType(), True),
    StructField('tip_amount', DoubleType(), True),
    StructField('tolls_amount', DoubleType(), True),
    StructField('improvement_surcharge', DoubleType(), True),
    StructField('total_amount', DoubleType(), True),
    StructField('congestion_surcharge', DoubleType(), True),
    StructField('Airport_fee', DoubleType(), True),
    StructField('cbd_congestion_fee', DoubleType(), True)
])

In [10]:
df_nytaxi = spark.read \
            .option("header", "true") \
            .schema(schema) \
            .parquet("yellow_tripdata_2025-11.parquet")
df_nytaxi.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

# Q2 Repartitioning 

In [12]:
#Q1 Repartitioning
df_nytaxi = df_nytaxi.repartition(4)
df_nytaxi.write.parquet('yellow-tripdata/2025/11/')

In [14]:
# check file size
folder_path = 'yellow-tripdata/2025/11/'

if os.path.exists(folder_path):
    for file in os.listdir(folder_path):
        if file.endswith(".parquet"):
            size_mb = os.path.getsize(os.path.join(folder_path, file)) / (1024 * 1024)
            print(f"{file}: {size_mb:.2f} MB")
else:
    print("Folder not found")

part-00000-6bfa7e11-91a3-4705-bbf5-ae267fee03c5-c000.snappy.parquet: 24.41 MB
part-00001-6bfa7e11-91a3-4705-bbf5-ae267fee03c5-c000.snappy.parquet: 24.41 MB
part-00002-6bfa7e11-91a3-4705-bbf5-ae267fee03c5-c000.snappy.parquet: 24.42 MB
part-00003-6bfa7e11-91a3-4705-bbf5-ae267fee03c5-c000.snappy.parquet: 24.42 MB


# Q3 Count Records

In [19]:
df_nytaxi = df_nytaxi.withColumnRenamed('tpep_pickup_datetime', 'pickup_datetime') \
                     .withColumnRenamed('tpep_dropoff_datetime', 'dropoff_datetime')

In [ ]:
df_nytaxi.createOrReplaceTempView("ny_taxi")

d:\DE-Zoomcamp-2026\06-batch\.venv\Lib\site-packages\pyspark\sql\classic\dataframe.py:178: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [ ]:
# How many taxi trips were there on the 15th of November?

spark.sql("""
SELECT COUNT(*) AS trip_count
FROM ny_taxi 
WHERE date(pickup_datetime) = '2025-11-15'
""").show()

+----------+
|trip_count|
+----------+
|    162604|
+----------+



# Q4 Longest Trip

In [37]:
# What is the length of the longest trip in the dataset in hours?

spark.sql("""
SELECT 
    ROUND((unix_timestamp(dropoff_datetime) - unix_timestamp(pickup_datetime)) / 3600.0, 2) AS diff_hours
FROM ny_taxi
ORDER BY diff_hours DESC
LIMIT 5
""").show()

+----------+
|diff_hours|
+----------+
|     90.65|
|     76.95|
|     76.21|
|     69.29|
|     67.08|
+----------+



# Q5 Spark UI

In [38]:
# Spark's User Interface which shows the application's dashboard runs on which local port?
print("4040")


4040


In [48]:
# Mendapatkan URL Spark UI yang aktif
ui_url = spark.sparkContext.uiWebUrl
print(f"Spark UI running on: {ui_url}")

Spark UI running on: http://LAPTOP-K9H7F3VK:4040


# Q6 Least Frequent Pickup Location Zone

In [40]:
!curl -o taxi_zone_lookup.csv https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:--  0:00:01 --:--:--     0
  0     0    0     0    0     0      0      0 --:--:--  0:00:02 --:--:--     0
100 12331  100 12331    0     0   3782      0  0:00:03  0:00:03 --:--:--  3784


In [42]:
# Read data from CSV
df_zones = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("taxi_zone_lookup.csv")

df_zones.show(5)

# TableView
df_zones.createOrReplaceTempView("zones")

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


In [46]:
# Using the zone lookup data and the Yellow November 2025 data, what is the name of the LEAST frequent pickup location Zone?

spark.sql("""
WITH zone_counts AS (
    SELECT 
        z.Zone, 
        COUNT(*) AS pickup_count
    FROM 
        ny_taxi t
    JOIN 
        zones z ON t.PULocationID = z.LocationID
    GROUP BY 
        z.Zone
),
ranked_zones AS (
    SELECT 
        Zone, 
        pickup_count,
        RANK() OVER (ORDER BY pickup_count ASC) as rnk
    FROM 
        zone_counts
)
SELECT 
    Zone, 
    pickup_count
FROM 
    ranked_zones
WHERE 
    rnk = 1;
""").show()

+--------------------+------------+
|                Zone|pickup_count|
+--------------------+------------+
|Governor's Island...|           1|
|       Arden Heights|           1|
|Eltingville/Annad...|           1|
+--------------------+------------+



In [16]:
df_nytaxi.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       2| 2025-11-04 21:00:59|  2025-11-04 21:09:12|              1|          0.0|         1|                 N|         162|    